# Example running Bayesian procedure with trained ML models

In [ ]:
import pickle
import numpy as np
from scipy.special import comb,factorial
from sensing_functions import create_dataset, create_model,sensing_protocol
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import matplotlib.patches as mpatches
from matplotlib.ticker import AutoMinorLocator
from matplotlib.gridspec import GridSpec
import pickle

In [ ]:
#1) Load Dataset

with open("Dataset_April_2025_sorted_90_10", "rb") as fl:
    training_x, training_y, testing_x, testing_y, dataset_x_sensing, dataset_y_sensing = [pickle.load(fl) for _ in range(6) ]

#2) Normalize the parameters for the ML 

tau_max = np.max([np.max(training_x[:,0]), np.max(testing_x[:,0])])
fb_max  = np.max([np.max(np.abs(training_x[:,1])), np.max(np.abs(testing_x[:,1]))])
fb_min  = -fb_max
phi_max = 2*np.pi
T_max  = np.max([np.max(training_x[:,3]), np.max(testing_x[:,3])])
P0_max = np.max([np.max(training_x[:,4:]), np.max(testing_x[:,4:])])

training_x = np.concatenate([training_x[:, 0:1]/tau_max, training_x[:, 1:2]/fb_max, training_x[:, 2:3]/phi_max, training_x[:, 3:4]/T_max,  training_x[:, 4:] ], axis=-1)
testing_x  = np.concatenate([testing_x[:,  0:1]/tau_max,  testing_x[:, 1:2]/fb_max,  testing_x[:, 2:3]/phi_max,  testing_x[:, 3:4]/T_max,   testing_x[:, 4:] ], axis=-1)

Pcl_max    = np.max([np.max(training_y), np.max(testing_y)])
training_y = training_y/Pcl_max
testing_y  = testing_y/Pcl_max

maximums   = [tau_max, fb_max, phi_max, T_max, P0_max, Pcl_max]
print(training_x.shape)
print(training_y.shape)
print(testing_x.shape)
print(testing_y.shape)

R     = 5e6
T2    = 5.4
F     = np.linspace(fb_min, fb_max, 5000)
uniform_prior  = np.ones((len(F),))


mu_f    = 0.5*(fb_min + fb_max)
sigma_f = ( fb_max - fb_min)/6
Gaussian_prior = np.array( [np.exp(-0.5*( (f-mu_f)**2)/ (sigma_f**2))/np.sqrt(2*np.pi*(sigma_f**2)) for f in F]) 

num_permutations = 100

In [ ]:
box        = "WB"
model_name = "model_WB_exp_14_07_2025"
model_WB   = create_model(box, model_name, maximums)  
model_WB.model.set_weights([np.array(1/(T2))])

box        = "GB"
model_name = "model_GB_exp_11_08_2026"
model_GB   = create_model(box, model_name, maximums)

box        = "BB_small"
model_name = "model_BB_small_exp_12_08_2026"
model_BB   = create_model(box, model_name, maximums)

In [ ]:
print("%e"%model_BB.training_history[-1])
print("%e"%model_BB.val_history[-1])

In [ ]:
print("%e"%model_GB.training_history[-1])
print("%e"%model_GB.val_history[-1])

In [ ]:
plt.figure(figsize=[4.75,4])
plt.loglog(np.arange(1,(1e5)+1), model_GB.training_history, label="GB Training", color="C1")
plt.loglog(np.arange(1,(1e5)+1), model_GB.val_history, label="GB Testing", color="C1", linestyle="--")


plt.loglog(np.arange(1,(1e5)+1), model_BB.training_history, label="BB Training", color="C2")
plt.loglog(np.arange(1,(1e5)+1), model_BB.val_history, label="BB Testing",  color="C2", linestyle="--")

plt.grid()
plt.xlabel("Iterations", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.savefig("Model_Training.pdf", format='pdf', bbox_inches='tight')

In [ ]:
posterior_uniform = []
posterior_Gauss   = []
for idx_key in range(len(dataset_x_sensing)):
    n_Sensing        = dataset_x_sensing[idx_key].shape[0]
    idx_permutations = [np.random.permutation(n_Sensing) for _ in range(num_permutations)]
    sensing_protocol(dataset_x_sensing, dataset_y_sensing, idx_key, F, uniform_prior, "uniform", model_WB, model_GB, model_BB, save=False, idx_permutations=idx_permutations)
    sensing_protocol(dataset_x_sensing, dataset_y_sensing, idx_key, F, Gaussian_prior, "Gauss",  model_WB, model_GB, model_BB, save=False, idx_permutations=idx_permutations)

In [ ]:
def get_one_cali(prior_name, idx_perm):
    cali_list_WB = np.zeros((2,len(dataset_x_sensing))) 
    cali_list_GB = np.zeros((2,len(dataset_x_sensing))) 
    cali_list_BB = np.zeros((2,len(dataset_x_sensing))) 
    
    for idx in range(len(dataset_x_sensing)):
        x = np.load("Results_%s_%d.npz"%(prior_name, idx))
        cali_list_WB[:, idx] =  [x["est_WB"][idx_perm,-1], dataset_x_sensing[idx][0, 1]]
        cali_list_GB[:, idx] =  [x["est_GB"][idx_perm,-1], dataset_x_sensing[idx][0, 1]]
        cali_list_BB[:, idx] =  [x["est_BB"][idx_perm,-1], dataset_x_sensing[idx][0, 1]]
    
    cali_list_WB = np.concatenate([cali_list_WB[:,idx:idx+1] for idx in np.argsort(cali_list_WB[0,:])], 1)
    cali_list_GB = np.concatenate([cali_list_GB[:,idx:idx+1] for idx in np.argsort(cali_list_GB[0,:])], 1)
    cali_list_BB = np.concatenate([cali_list_BB[:,idx:idx+1] for idx in np.argsort(cali_list_BB[0,:])], 1)
    
    cali_x_WB = np.zeros((10,1))
    cali_y_WB = np.zeros((10,1))
    cali_x_GB = np.zeros((10,1))
    cali_y_GB = np.zeros((10,1))
    cali_x_BB = np.zeros((10,1))
    cali_y_BB = np.zeros((10,1))
    
    for idx in range(9):
        cali_x_WB[idx] = np.mean(cali_list_WB[0, idx*16:idx*16 + 16])
        cali_y_WB[idx] = np.mean(cali_list_WB[1, idx*16:idx*16 + 16])
        cali_x_GB[idx] = np.mean(cali_list_GB[0, idx*16:idx*16 + 16])
        cali_y_GB[idx] = np.mean(cali_list_GB[1, idx*16:idx*16 + 16])
        cali_x_BB[idx] = np.mean(cali_list_BB[0, idx*16:idx*16 + 16])
        cali_y_BB[idx] = np.mean(cali_list_BB[1, idx*16:idx*16 + 16])
    
    idx=9
    cali_x_WB[idx] = np.mean(cali_list_WB[0, idx*16:])
    cali_y_WB[idx] = np.mean(cali_list_WB[1, idx*16:])
    cali_x_GB[idx] = np.mean(cali_list_GB[0, idx*16:])
    cali_y_GB[idx] = np.mean(cali_list_GB[1, idx*16:])
    cali_x_BB[idx] = np.mean(cali_list_BB[0, idx*16:])
    cali_y_BB[idx] = np.mean(cali_list_BB[1, idx*16:])

    return cali_x_WB, cali_y_WB, cali_x_GB, cali_y_GB, cali_x_BB, cali_y_BB

## Uniform Prior Bayesian update

In [ ]:
mse_GB_final = []
mse_WB_final = []
mse_BB_final = []

var_GB_final = []
var_WB_final = []
var_BB_final = []

for idx_key in range(len(dataset_x_sensing)):
    x = np.load("Results_uniform_%d.npz"%idx_key)
    mse_WB_final.append(np.mean(x["mse_WB"][:,-1]))
    mse_GB_final.append(np.mean(x["mse_GB"][:,-1]))
    mse_BB_final.append(np.mean(x["mse_BB"][:,-1]))
    var_WB_final.append(np.mean(x["var_WB"][:,-1]))
    var_GB_final.append(np.mean(x["var_GB"][:,-1]))
    var_BB_final.append(np.mean(x["var_BB"][:,-1]))

idx_sort    = np.argsort(mse_GB_final)
idx_best, idx_average, idx_worst  = idx_sort[0], idx_sort[1 + len(dataset_x_sensing)//2], idx_sort[-1]
idx_list = [idx_best, idx_average, idx_worst]

est_WB = []
est_GB = []
est_BB = []

mse_WB = []
mse_GB = []
mse_BB = []

var_WB = []
var_GB = []
var_BB = []

posterior_uniform = []
for idx in idx_list:
    x =  np.load("Results_uniform_%d.npz"%idx)
    est_WB.append(x["est_WB"])
    est_GB.append(x["est_GB"])
    est_BB.append(x["est_BB"])
    mse_WB.append(x["mse_WB"])
    mse_GB.append(x["mse_GB"])
    mse_BB.append(x["mse_BB"])
    
    var_GB.append(x["var_GB"])
    var_WB.append(x["var_WB"])
    var_BB.append(x["var_BB"])

    fl = open("posterior_uniform_%d.pckl"%idx, 'rb')
    posterior_uniform.append( pickle.load(fl)["posterior"] )
    fl.close()

In [ ]:
cali_x_WB, cali_y_WB, cali_x_GB, cali_y_GB, cali_x_BB, cali_y_BB = [ [] for _ in range(6)]
for idx_perm in range(100):
    x = get_one_cali("uniform", idx_perm)
    cali_x_WB.append(x[0])
    cali_y_WB.append(x[1])
    cali_x_GB.append(x[2])
    cali_y_GB.append(x[3])
    cali_x_BB.append(x[4])
    cali_y_BB.append(x[5]) 
    
cali_x_WB = np.concatenate(cali_x_WB, 1).T
cali_y_WB = np.concatenate(cali_y_WB, 1).T
cali_x_GB = np.concatenate(cali_x_GB, 1).T
cali_y_GB = np.concatenate(cali_y_GB, 1).T
cali_x_BB = np.concatenate(cali_x_BB, 1).T
cali_y_BB = np.concatenate(cali_y_BB, 1).T

In [ ]:
trans = mtransforms.ScaledTranslation(-20/72, 7/72, plt.gcf().dpi_scale_trans)

plt.figure(figsize=[12,3.5])

plt.subplot(1,3,1)
plt.text(0.0, 1.0, "a)", weight="bold", transform=plt.gca().transAxes + trans) 
violin_plot = plt.violinplot(np.log10(mse_WB_final), [1], showmeans=False,showmedians=True)
violin_plot = plt.violinplot(np.log10(mse_GB_final), [2], showmeans=False,showmedians=True)
violin_plot = plt.violinplot(np.log10(mse_BB_final), [3], showmeans=False,showmedians=True)

#plt.legend()#zip(*labels))
plt.gca().set_yticks(np.arange(-12, 3, 2))
plt.gca().set_yticklabels(["%.0e"%10.0**idx for idx in np.arange(-12, 3, 2)])
plt.ylim([-12.5,2.5])
plt.setp(plt.gca(), xticks=np.array([1,2,3]), xticklabels=["WB","GB","BB"])
plt.ylabel(r"Final MSE$(f_b,\hat{f}_b)$")
plt.grid()

plt.subplot(1,3,2)
plt.text(0.0, 1.0, "b)", weight="bold", transform=plt.gca().transAxes + trans) 
plt.violinplot(np.log10(var_WB_final), [1], showmeans=False,showmedians=True)
plt.violinplot(np.log10(var_GB_final), [2], showmeans=False,showmedians=True)
plt.violinplot(np.log10(var_WB_final), [3], showmeans=False,showmedians=True)
plt.gca().set_yticks(np.arange(-15, 3, 2))
plt.gca().set_yticklabels(["%.0e"%10.0**idx for idx in np.arange(-15, 3, 2)])
plt.ylim([-15.5,2.5])
plt.setp(plt.gca(), xticks=np.array([1,2,3]), xticklabels=["WB","GB", "BB"])
plt.ylabel(r"Final MSE$(f_b,\hat{f}_b)$")
plt.grid()
plt.ylabel(r"Final Variance$(f_b,\hat{f}_b)$")

plt.subplot(1,3,3)
plt.text(0.0, 1.0, "c)", weight="bold", transform=plt.gca().transAxes + trans) 
plt.errorbar(np.mean(cali_x_WB, 0), np.mean(cali_y_WB, 0),  [np.median(cali_x_WB, 0) - np.percentile(cali_x_WB, 2.5, 0), np.percentile(cali_x_WB, 97.5, 0) - np.median(cali_x_WB, 0)], [np.median(cali_y_WB, 0) - np.percentile(cali_y_WB, 2.5, 0), np.percentile(cali_y_WB, 97.5, 0) - np.median(cali_y_WB, 0)], fmt = "--", label="WB", capsize=3)
plt.errorbar(np.mean(cali_x_GB, 0), np.mean(cali_y_GB, 0),  [np.median(cali_x_GB, 0) - np.percentile(cali_x_GB, 2.5, 0), np.percentile(cali_x_GB, 97.5, 0) - np.median(cali_x_GB, 0)], [np.median(cali_y_GB, 0) - np.percentile(cali_y_GB, 2.5, 0), np.percentile(cali_y_GB, 97.5, 0) - np.median(cali_y_GB, 0)], fmt ="--",  label="GB", capsize=3)
plt.errorbar(np.mean(cali_x_BB, 0), np.mean(cali_y_BB, 0),  [np.median(cali_x_BB, 0) - np.percentile(cali_x_BB, 2.5, 0), np.percentile(cali_x_BB, 97.5, 0) - np.median(cali_x_BB, 0)], [np.median(cali_y_BB, 0) - np.percentile(cali_y_BB, 2.5, 0), np.percentile(cali_y_BB, 97.5, 0) - np.median(cali_y_BB, 0)], fmt = "--", label="BB", capsize=3)
plt.plot(np.arange(-4,5.5), np.arange(-4,5.5), "k--", label="Perfect")
plt.xlabel(r"Predicted $\hat{f}_b$ (MHZ)")
plt.ylabel(r"True $f_b$ (MHz)")
plt.legend(ncol=2)
plt.grid()

plt.tight_layout()
plt.savefig("Stats_uniform.pdf", format='pdf', bbox_inches='tight')

In [ ]:
labels_1 = ["a)", "b)", "c)"]
labels_2 = ["d)", "e)", "f)"]

plt.figure(figsize=[12, 6])
for idx in range(3):
    n_Sensing        = dataset_x_sensing[idx_list[idx]].shape[0]
    fb_true          = dataset_x_sensing[idx_list[idx]][0, 1]
    
    plt.subplot(2,3,idx+1)
    plt.text(0.0, 1.0, labels_1[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h1 = plt.loglog(range(1, n_Sensing+2), np.mean(mse_WB[idx], 0), label="WB")
    h2 = plt.loglog(range(1, n_Sensing+2), np.mean(mse_GB[idx], 0), label="GB")
    h3 = plt.loglog(range(1, n_Sensing+2), np.mean(mse_BB[idx], 0), label="BB")
    plt.xlabel('Iteration')
    plt.ylabel('MSE$(\hat{f}_B, f_B)$ (MHZ)$^2$')
    plt.ylim([1e-12,1e2])
    plt.xlim([1,250])
    plt.grid(which="both")
    
    plt.subplot(2,3,3+idx+1)
    plt.text(0.0, 1.0, labels_2[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    plt.plot(range(1, n_Sensing+2), np.mean(est_WB[idx], 0))
    plt.fill_between(range(1, n_Sensing+2), np.mean(est_WB[idx], 0) - np.mean(np.sqrt(var_WB[idx]), 0)*0.5, np.mean(est_WB[idx], 0) + np.mean(np.sqrt(var_WB[idx]), 0)*0.5, alpha=0.1)
    
    plt.plot(range(1, n_Sensing+2), np.mean(est_GB[idx],0))
    plt.fill_between(range(1, n_Sensing+2), np.mean(est_GB[idx], 0) - np.mean(np.sqrt(var_GB[idx]), 0)*0.5, np.mean(est_GB[idx], 0) + np.mean(np.sqrt(var_GB[idx]), 0)*0.5, alpha=0.1)
    
    plt.plot(range(1, n_Sensing+2), np.mean(est_BB[idx],0))
    plt.fill_between(range(1, n_Sensing+2), np.mean(est_BB[idx], 0) - np.mean(np.sqrt(var_BB[idx]), 0)*0.5, np.mean(est_BB[idx], 0) + np.mean(np.sqrt(var_BB[idx]), 0)*0.5, alpha=0.1)

    h4 = plt.axhline(fb_true, linestyle="--", color="black", label="True")
    plt.xlabel('Iteration')
    plt.ylabel(r'$f_B$ (MHz)')
    plt.gca().set_xticks(np.arange(0, n_Sensing+2, n_Sensing/4))
    #plt.xlim([0,130])
    plt.grid()

plt.subplots_adjust(hspace=0.75, wspace=0.4)
plt.gcf().legend(handles = [h1[0],h2[0],h3[0],h4], labels=["WB","GB","BB","True"], loc="center", ncol = 4)

plt.savefig("Fig5_uniform.pdf", format='pdf', bbox_inches='tight')

## Gaussian Prior Bayesian Update

In [ ]:
mse_GB_final = []
mse_WB_final = []
mse_BB_final = []

var_GB_final = []
var_WB_final = []
var_BB_final = []

for idx_key in range(len(dataset_x_sensing)):
    x = np.load("Results_Gauss_%d.npz"%idx_key)
    mse_WB_final.append(np.mean(x["mse_WB"][:,-1]))
    mse_GB_final.append(np.mean(x["mse_GB"][:,-1]))
    mse_BB_final.append(np.mean(x["mse_BB"][:,-1]))
    var_WB_final.append(np.mean(x["var_WB"][:,-1]))
    var_GB_final.append(np.mean(x["var_GB"][:,-1]))
    var_BB_final.append(np.mean(x["var_BB"][:,-1]))

idx_sort    = np.argsort(mse_GB_final)
idx_best, idx_average, idx_worst  = idx_sort[0], idx_sort[1 + len(dataset_x_sensing)//2], idx_sort[-1]
idx_list = [idx_best, idx_average, idx_worst]

est_WB = []
est_GB = []
est_BB = []

mse_WB = []
mse_GB = []
mse_BB = []

var_WB = []
var_GB = []
var_BB = []

posterior_Gauss = []

for idx in idx_list:
    x =  np.load("Results_Gauss_%d.npz"%idx)
    est_WB.append(x["est_WB"])
    est_GB.append(x["est_GB"])
    est_BB.append(x["est_BB"])
    mse_WB.append(x["mse_WB"])
    mse_GB.append(x["mse_GB"])
    mse_BB.append(x["mse_BB"])
    
    var_GB.append(x["var_GB"])
    var_WB.append(x["var_WB"])
    var_BB.append(x["var_BB"])

    fl = open("posterior_Gauss_%d.pckl"%idx, 'rb')
    posterior_Gauss.append( pickle.load(fl)["posterior"] )
    fl.close()

In [ ]:
cali_x_WB, cali_y_WB, cali_x_GB, cali_y_GB, cali_x_BB, cali_y_BB = [ [] for _ in range(6)]
for idx_perm in range(100):
    x = get_one_cali("Gauss", idx_perm)
    cali_x_WB.append(x[0])
    cali_y_WB.append(x[1])
    cali_x_GB.append(x[2])
    cali_y_GB.append(x[3])
    cali_x_BB.append(x[4])
    cali_y_BB.append(x[5]) 
    
cali_x_WB = np.concatenate(cali_x_WB, 1).T
cali_y_WB = np.concatenate(cali_y_WB, 1).T
cali_x_GB = np.concatenate(cali_x_GB, 1).T
cali_y_GB = np.concatenate(cali_y_GB, 1).T
cali_x_BB = np.concatenate(cali_x_BB, 1).T
cali_y_BB = np.concatenate(cali_y_BB, 1).T

In [ ]:
trans = mtransforms.ScaledTranslation(-20/72, 7/72, plt.gcf().dpi_scale_trans)

plt.figure(figsize=[12,3.5])

plt.subplot(1,3,1)
plt.text(0.0, 1.0, "a)", weight="bold", transform=plt.gca().transAxes + trans) 
violin_plot = plt.violinplot(np.log10(mse_WB_final), [1], showmeans=False,showmedians=True)
violin_plot = plt.violinplot(np.log10(mse_GB_final), [2], showmeans=False,showmedians=True)
violin_plot = plt.violinplot(np.log10(mse_BB_final), [3], showmeans=False,showmedians=True)

#plt.legend()#zip(*labels))
plt.gca().set_yticks(np.arange(-12, 3, 2))
plt.gca().set_yticklabels(["%.0e"%10.0**idx for idx in np.arange(-12, 3, 2)])
plt.ylim([-12.5,2.5])
plt.setp(plt.gca(), xticks=np.array([1,2,3]), xticklabels=["WB","GB","BB"])
plt.ylabel(r"Final MSE$(f_b,\hat{f}_b)$")
plt.grid()

plt.subplot(1,3,2)
plt.text(0.0, 1.0, "b)", weight="bold", transform=plt.gca().transAxes + trans) 
plt.violinplot(np.log10(var_WB_final), [1], showmeans=False,showmedians=True)
plt.violinplot(np.log10(var_GB_final), [2], showmeans=False,showmedians=True)
plt.violinplot(np.log10(var_WB_final), [3], showmeans=False,showmedians=True)
plt.gca().set_yticks(np.arange(-15, 3, 2))
plt.gca().set_yticklabels(["%.0e"%10.0**idx for idx in np.arange(-15, 3, 2)])
plt.ylim([-15.5,2.5])
plt.setp(plt.gca(), xticks=np.array([1,2,3]), xticklabels=["WB","GB", "BB"])
plt.ylabel(r"Final MSE$(f_b,\hat{f}_b)$")
plt.grid()
plt.ylabel(r"Final Variance$(f_b,\hat{f}_b)$")

plt.subplot(1,3,3)
plt.text(0.0, 1.0, "c)", weight="bold", transform=plt.gca().transAxes + trans) 
plt.errorbar(np.mean(cali_x_WB, 0), np.mean(cali_y_WB, 0),  [np.median(cali_x_WB, 0) - np.percentile(cali_x_WB, 2.5, 0), np.percentile(cali_x_WB, 97.5, 0) - np.median(cali_x_WB, 0)], [np.median(cali_y_WB, 0) - np.percentile(cali_y_WB, 2.5, 0), np.percentile(cali_y_WB, 97.5, 0) - np.median(cali_y_WB, 0)], fmt = "--", label="WB", capsize=3)
plt.errorbar(np.mean(cali_x_GB, 0), np.mean(cali_y_GB, 0),  [np.median(cali_x_GB, 0) - np.percentile(cali_x_GB, 2.5, 0), np.percentile(cali_x_GB, 97.5, 0) - np.median(cali_x_GB, 0)], [np.median(cali_y_GB, 0) - np.percentile(cali_y_GB, 2.5, 0), np.percentile(cali_y_GB, 97.5, 0) - np.median(cali_y_GB, 0)], fmt ="--",  label="GB", capsize=3)
plt.errorbar(np.mean(cali_x_BB, 0), np.mean(cali_y_BB, 0),  [np.median(cali_x_BB, 0) - np.percentile(cali_x_BB, 2.5, 0), np.percentile(cali_x_BB, 97.5, 0) - np.median(cali_x_BB, 0)], [np.median(cali_y_BB, 0) - np.percentile(cali_y_BB, 2.5, 0), np.percentile(cali_y_BB, 97.5, 0) - np.median(cali_y_BB, 0)], fmt = "--", label="BB", capsize=3)
plt.plot(np.arange(-4,5.5), np.arange(-4,5.5), "k--", label="Perfect")
plt.xlabel(r"Predicted $\hat{f}_b$ (MHZ)")
plt.ylabel(r"True $f_b$ (MHz)")
plt.legend(ncol=2)
plt.grid()

plt.tight_layout()
plt.savefig("Stats_Gauss.pdf", format='pdf', bbox_inches='tight')

In [ ]:
labels_1 = ["a)", "b)", "c)"]
labels_2 = ["d)", "e)", "f)"]

plt.figure(figsize=[12, 6])
for idx in range(3):
    n_Sensing        = dataset_x_sensing[idx_list[idx]].shape[0]
    fb_true          = dataset_x_sensing[idx_list[idx]][0, 1]
    
    plt.subplot(2,3,idx+1)
    plt.text(0.0, 1.0, labels_1[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h1 = plt.loglog(range(1, n_Sensing+2), np.mean(mse_WB[idx], 0), label="WB")
    h2 = plt.loglog(range(1, n_Sensing+2), np.mean(mse_GB[idx], 0), label="GB")
    h3 = plt.loglog(range(1, n_Sensing+2), np.mean(mse_BB[idx], 0), label="BB")
    plt.xlabel('Iteration')
    plt.ylabel('MSE$(\hat{f}_B, f_B)$ (MHZ)$^2$')
    plt.ylim([1e-12,1e2])
    plt.xlim([1,250])
    plt.grid(which="both")
    
    plt.subplot(2,3,3+idx+1)
    plt.text(0.0, 1.0, labels_2[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    plt.plot(range(1, n_Sensing+2), np.mean(est_WB[idx], 0))
    plt.fill_between(range(1, n_Sensing+2), np.mean(est_WB[idx], 0) - np.mean(np.sqrt(var_WB[idx]), 0)*0.5, np.mean(est_WB[idx], 0) + np.mean(np.sqrt(var_WB[idx]), 0)*0.5, alpha=0.1)
    
    plt.plot(range(1, n_Sensing+2), np.mean(est_GB[idx],0))
    plt.fill_between(range(1, n_Sensing+2), np.mean(est_GB[idx], 0) - np.mean(np.sqrt(var_GB[idx]), 0)*0.5, np.mean(est_GB[idx], 0) + np.mean(np.sqrt(var_GB[idx]), 0)*0.5, alpha=0.1)
    
    plt.plot(range(1, n_Sensing+2), np.mean(est_BB[idx],0))
    plt.fill_between(range(1, n_Sensing+2), np.mean(est_BB[idx], 0) - np.mean(np.sqrt(var_BB[idx]), 0)*0.5, np.mean(est_BB[idx], 0) + np.mean(np.sqrt(var_BB[idx]), 0)*0.5, alpha=0.1)

    h4 = plt.axhline(fb_true, linestyle="--", color="black", label="True")
    plt.xlabel('Iteration')
    plt.ylabel(r'$f_B$ (MHz)')
    plt.gca().set_xticks(np.arange(0, n_Sensing+2, n_Sensing/4))
    #plt.xlim([0,130])
    plt.grid()

plt.subplots_adjust(hspace=0.75, wspace=0.4)
plt.gcf().legend(handles = [h1[0],h2[0],h3[0],h4], labels=["WB","GB","BB","True"], loc="center", ncol = 4)

plt.savefig("Fig5_Gauss.pdf", format='pdf', bbox_inches='tight')

In [ ]:
titles_it = ["Prior", "Posterior: First Iteration", "Posterior: Last Iteration"]
it = [0,1,-1]
plt.figure(figsize=[12, 6])

fb_true = dataset_x_sensing[idx_list[0]][0, 1]
for idx in range(3):   
    plt.subplot(2,3, idx+1)
    h1=plt.plot(F, posterior_uniform[0][0][it[idx]], label="WB")
    h2=plt.plot(F, posterior_uniform[0][1][it[idx]], label="GB")
    h3=plt.plot(F, posterior_uniform[0][2][it[idx]], label="BB")
    plt.grid()
    plt.xlabel(r"$f_b$ (MHz)")
    plt.ylabel("Distribution")
    plt.title(titles_it[idx])
    plt.text(0.0, 1.0, labels_1[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h4 = plt.axvline(fb_true, linestyle="--", color="black", label="True")
    if(idx==2):
        plt.xlim([-2,2])
    
    plt.subplot(2,3, idx+3+1)
    plt.plot(F, posterior_Gauss[0][0][it[idx]], label="WB")
    plt.plot(F, posterior_Gauss[0][1][it[idx]], label="GB")
    plt.plot(F, posterior_Gauss[0][2][it[idx]], label="BB")
    plt.grid()
    plt.xlabel(r"$f_b$ (MHz)")
    plt.ylabel("Distribution")
    plt.title(titles_it[idx])
    plt.text(0.0, 1.0, labels_2[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h4 = plt.axvline(fb_true, linestyle="--", color="black", label="True")
    if(idx==2):
        plt.xlim([-2,2])

plt.subplots_adjust(hspace=0.75, wspace=0.4)
plt.gcf().legend(handles = [h1[0],h2[0],h3[0],h4], labels=["WB","GB","BB","True"], loc="center", ncol = 4)

plt.savefig("Posterior_good.pdf", format='pdf', bbox_inches='tight')

In [ ]:
titles_it = ["Prior", "Posterior: First Iteration", "Posterior: Last Iteration"]
it = [0,1,-1]
plt.figure(figsize=[12, 6])

fb_true = dataset_x_sensing[idx_list[1]][0, 1]
for idx in range(3):   
    plt.subplot(2,3, idx+1)
    h1=plt.plot(F, posterior_uniform[1][0][it[idx]], label="WB")
    h2=plt.plot(F, posterior_uniform[1][1][it[idx]], label="GB")
    h3=plt.plot(F, posterior_uniform[1][2][it[idx]], label="BB")
    plt.grid()
    plt.xlabel(r"$f_b$ (MHz)")
    plt.ylabel("Distribution")
    plt.title(titles_it[idx])
    plt.text(0.0, 1.0, labels_1[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h4 = plt.axvline(fb_true, linestyle="--", color="black", label="True")
    if(idx==2):
        plt.xlim([-2.5,3])

    plt.subplot(2,3, idx+3+1)
    plt.plot(F, posterior_Gauss[1][0][it[idx]], label="WB")
    plt.plot(F, posterior_Gauss[1][1][it[idx]], label="GB")
    plt.plot(F, posterior_Gauss[1][2][it[idx]], label="BB")
    plt.grid()
    plt.xlabel(r"$f_b$ (MHz)")
    plt.ylabel("Distribution")
    plt.title(titles_it[idx])
    plt.text(0.0, 1.0, labels_2[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h4 = plt.axvline(fb_true, linestyle="--", color="black", label="True")
    if(idx==2):
        plt.xlim([-2.5,3])


plt.subplots_adjust(hspace=0.75, wspace=0.4)
plt.gcf().legend(handles = [h1[0],h2[0],h3[0],h4], labels=["WB","GB","BB","True"], loc="center", ncol = 4)

plt.savefig("Posterior_avg.pdf", format='pdf', bbox_inches='tight')

In [ ]:
titles_it = ["Prior", "Posterior: First Iteration", "Posterior: Last Iteration"]

plt.figure(figsize=[12, 6])

fb_true = dataset_x_sensing[idx_list[2]][0, 1]

it = [0,1,-1]
for idx in range(3):   
    plt.subplot(2,3, idx+1)
    h1=plt.plot(F, posterior_uniform[2][0][it[idx]], label="WB")
    h2=plt.plot(F, posterior_uniform[2][1][it[idx]], label="GB")
    h3=plt.plot(F, posterior_uniform[2][2][it[idx]], label="BB")
    plt.grid()
    plt.xlabel(r"$f_b$ (MHz)")
    plt.ylabel("Distribution")
    plt.title(titles_it[idx])
    plt.text(0.0, 1.0, labels_1[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h4 = plt.axvline(fb_true, linestyle="--", color="black", label="True")
    if(idx==2):
        plt.xlim([-0.5,4.5])
        
    plt.subplot(2,3, idx+3+1)
    plt.plot(F, posterior_Gauss[2][0][it[idx]], label="WB")
    plt.plot(F, posterior_Gauss[2][1][it[idx]], label="GB")
    plt.plot(F, posterior_Gauss[2][2][it[idx]], label="BB")
    plt.grid()
    plt.xlabel(r"$f_b$ (MHz)")
    plt.ylabel("Distribution")
    plt.title(titles_it[idx])
    plt.text(0.0, 1.0, labels_2[idx], weight="bold", transform=plt.gca().transAxes + trans) 
    h4 = plt.axvline(fb_true, linestyle="--", color="black", label="True")
    if(idx==2):
        plt.xlim([-0.5,4.5])

plt.subplots_adjust(hspace=0.75, wspace=0.4)
plt.gcf().legend(handles = [h1[0],h2[0],h3[0],h4], labels=["WB","GB","BB","True"], loc="center", ncol = 4)

plt.savefig("Posterior_bad.pdf", format='pdf', bbox_inches='tight')